In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
import torch
import copy
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
from tqdm import tqdm

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from src.trainer import SimpleTrainer, SITrainer, IntervalTrainer
from src.data_utils import (
    get_mnist_tasks,
    _extract_targets,
    get_context_sets,
    create_holdout_set,
)
from src.utils.general import InContextHead
from src import models
from src.regulariser import MultiRegulariser, L2Regulariser, UnbiasRegulariser

### Class Incremental Learning

In [4]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [ ]:
model = models.get_mnist_model(device="cuda")

trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=1, batch_size=300)
trainer.test(test_tasks[0:2])

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="CIL",
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], si_batch=samples, prune_prop=0.8)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256)
    si_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(test, si_batch=samples, prune_prop=0.8)

### Task Incremental Learning

In [17]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

context_sets = get_context_sets(test_tasks)
head = InContextHead(context_sets, 10, device="cuda")
head.set_context(0)
model = models.get_mnist_model(head=head, device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [18]:
trainer = SimpleTrainer(model)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="TIL",
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(
    test_tasks[0], si_batch=samples, prune_prop=0.8, context_id=0, si_context_id=1
)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256, context_id=i)
    si_trainer.test(test_tasks[0 : i + 1], context_list=list(range(i + 1)))
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(
            test, si_batch=samples, prune_prop=0.8, context_id=i, si_context_id=i + 1
        )

### Domain Incremental Learning

In [20]:
train_tasks, val_tasks, test_tasks = get_mnist_tasks()

model = models.get_mnist_model(device="cuda")

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

In [21]:
def domain_map_fn(labels: torch.Tensor) -> torch.Tensor:
    """Map the global label to the in context label."""
    return labels % 2

In [22]:
trainer = SimpleTrainer(model, paradigm="DIL", domain_map_fn=domain_map_fn)
trainer.train(train_tasks[0], val_tasks[0], epochs=3, batch_size=256)
trainer.test(test_tasks[0:1])

In [ ]:
si_trainer = SITrainer(
    trainer.model,
    checkpoint=20,
    n_iters=200,
    min_acc_limit=0.7,
    primal_learning_rate=0.5,
    paradigm="DIL",
    domain_map_fn=domain_map_fn,
)

loader = DataLoader(
    train_tasks[1],
    batch_size=128,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
samples = next(iter(loader))
# Compute bounds for task 0
si_trainer.compute_rashomon_set(test_tasks[0], si_batch=samples, prune_prop=0.8)

for i, (train, val, test) in enumerate(
    zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
):
    si_trainer.train(train, val, epochs=3, batch_size=256)
    si_trainer.test(test_tasks[0 : i + 1])
    if i < len(train_tasks) - 2:
        loader = DataLoader(
            train_tasks[i + 1],
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(42),
        )
        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(test, si_batch=samples, prune_prop=0.8)

### Experiments

In [5]:
SEED = 0

train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

print(
    f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
)

model = models.get_mnist_model(device="cuda", seed=SEED)

In [6]:
simple_trainer = SimpleTrainer(model, paradigm="CIL", seed=SEED)
simple_trainer.train(train_tasks[0], val_tasks[0])
simple_trainer.test(test_tasks)

In [7]:
interval_trainer = IntervalTrainer(
    simple_trainer.model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=SEED,
)

interval_trainer.compute_rashomon_set(test_tasks[0])
interval_trainer.train(train_tasks[1], val_tasks[1])
interval_trainer.test(test_tasks)

In [24]:
si_trainer = SITrainer(
    simple_trainer.model,
    checkpoint=200,
    n_iters=200,
    min_acc_limit=0.8,
    min_acc_increment=0,
    primal_learning_rate=0.5,
    paradigm="CIL",
    seed=SEED,
)

loader = DataLoader(
    train_tasks[0],
    batch_size=256,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
samples = next(iter(loader))
si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=0.9, si_batch=samples, preserve_top=True)
si_trainer.train(train_tasks[1], val_tasks[1])
si_trainer.test(test_tasks)

In [9]:
def pretrain(seed: int = 42, use_regulariser: bool = True):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    unbias = UnbiasRegulariser(
        lmbd=0.01,
        unbias_domain=[
            torch.zeros(1, 1, 28, 28, device="cuda"),
            torch.ones(1, 1, 28, 28, device="cuda"),
        ],
    )
    l2 = L2Regulariser(lmbd=0.01)
    regulariser = MultiRegulariser([l2, unbias])

    model = models.get_mnist_model(device="cuda", seed=seed)

    simple_trainer = SimpleTrainer(model, paradigm="CIL", seed=seed)
    simple_trainer.train(train_tasks[0], val_tasks[0], regulariser=regulariser if use_regulariser else None)
    results = simple_trainer.test(test_tasks)

    return simple_trainer.model, results[0][1] < 0.75

def run_baseline(model: torch.nn.Module, seed: int = 42):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    interval_trainer = IntervalTrainer(
        model,
        checkpoint=20,
        n_iters=200,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    unbias = UnbiasRegulariser(
        lmbd=0.01,
        unbias_domain=[
            torch.zeros(1, 1, 28, 28, device="cuda"),
            torch.ones(1, 1, 28, 28, device="cuda"),
        ],
    )
    l2 = L2Regulariser(lmbd=0.01)
    regulariser = MultiRegulariser([l2, unbias])

    # Test just for verification purposes
    interval_trainer.test(test_tasks)

    interval_trainer.compute_rashomon_set(test_tasks[0])
    interval_trainer.train(train_tasks[1], val_tasks[1], regulariser=regulariser)
    baseline_acc = interval_trainer.test(test_tasks)[1][1]

    return baseline_acc

def run_trial(model: torch.nn.Module, prune_prop: float, lookahead: bool = False, rashomon_iters: int = 200, seed: int = 42):
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=seed)

    print(
        f"Tasks: {[torch._unique(_extract_targets(train))[0].tolist() for train in train_tasks]}"
    )

    si_trainer = SITrainer(
        model,
        checkpoint=100,
        n_iters=300,
        min_acc_limit=0.8,
        min_acc_increment=0,
        primal_learning_rate=0.5,
        paradigm="CIL",
        seed=seed,
    )

    # Test just for verification purposes
    si_trainer.test(test_tasks)

    loader = DataLoader(
        train_tasks[0 if not lookahead else 1],
        batch_size=400,
        shuffle=True,
        generator=torch.Generator().manual_seed(seed),
    )
    samples = next(iter(loader))
    si_trainer.compute_rashomon_set(test_tasks[0], prune_prop=prune_prop, si_batch=samples, preserve_top=not lookahead)
    si_trainer.train(train_tasks[1], val_tasks[1])
    trial_acc = si_trainer.test(test_tasks)[1][1]

    return trial_acc

#### Test SI Parameter Freezing

In [5]:
import json
trial_results = defaultdict(list)

for seed in range(14, 15):
    unregularised_model, failed = pretrain(seed=seed, use_regulariser=False)
    if failed:
        continue
    for prune_prop in [0.5, 0.7, 0.8, 0.825, 0.85, 0.9, 0.95]:
        trial = run_trial(model=unregularised_model, prune_prop=prune_prop, lookahead=True, seed=seed)

        trial_results[prune_prop].append(trial)
    with open(f'si_lookahead_finding_{seed}.json', 'w') as f:
        json.dump(trial_results, f, indent=4)

In [8]:
BEST_PRUNING_PROP = 0.825

baseline_results = defaultdict(list)
trial_results = defaultdict(list)

MAX_SEED = 15
for seed in range(5, MAX_SEED):
    model, failed = pretrain(seed=seed)
    if failed:
        continue
    # unregularised_model, failed = pretrain(seed=seed, use_regulariser=False)
    # if failed:
    #     continue
    baseline = run_baseline(model, seed=seed)
    # trial = run_trial(model=unregularised_model, prune_prop=BEST_PRUNING_PROP, lookahead=True, seed=seed)

    baseline_results[BEST_PRUNING_PROP].append(baseline)
    # trial_results[BEST_PRUNING_PROP].append(trial)

with open(f'si_performance_baseline_{MAX_SEED}.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)
# with open(f'si_performance_trial_{MAX_SEED}.json', 'w') as f:
#     json.dump(trial_results, f, indent=4)

In [10]:
prev_perf = baseline_results[BEST_PRUNING_PROP]
new_perf = trial_results[BEST_PRUNING_PROP]

prev_acc = np.array(prev_perf).mean()
prev_std = np.array(prev_perf).std()
new_acc = np.array(new_perf).mean()
new_std = np.array(new_perf).std()

# 2. Create a prettier plot
# Use a more balanced figure size
fig, ax = plt.subplots(figsize=(7, 6))

# Define some nice colors
colors = ["C0", "C1"]  # Steel Blue and a nice Green

# Plot the bars with improved styling
bars = ax.bar(
    x=["Previous", "New"],
    height=[prev_acc, new_acc],
    yerr=[prev_std, new_std],
    color=colors,
    alpha=0.8,  # Make bars slightly transparent
    edgecolor="black",  # Add a crisp black edge
    capsize=10,  # THIS IS KEY: Adds caps to the error bars
    ecolor="black",  # Color of the error bar lines
    linewidth=1.5,
)

# 3. Add Labels, Title, and Grid for context
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Lookahead Pruning Performance", fontsize=16, pad=20)
ax.set_xticks(ticks=[0, 1], labels=["Baseline", "Lookahead Pruning"], fontsize=12)

# Add a subtle horizontal grid to make comparisons easier
ax.yaxis.grid(True, linestyle="--", which="major", color="grey", alpha=0.3)

# Remove the top and right spines for a cleaner look
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Set a dynamic Y-axis limit for better spacing
ax.set_ylim(0, max(new_acc, prev_acc) + max(new_std, prev_std) + 0.05)

# 4. Add data labels on top of the bars
# This shows the exact mean value on the plot
ax.bar_label(bars, fmt="{:.3f}", padding=5, fontsize=11, color="black")

# Ensure everything fits without overlapping
plt.tight_layout()

plt.savefig("figures/fisher_performance.pdf", dpi=300)
plt.show()

In [ ]:
trial_results = defaultdict(list)

for seed in range(15):
    unregularised_model, failed= pretrain(seed=seed, use_regulariser=False)
    if failed:
        continue
    for prune_prop in [0.5, 0.7, 0.8, 0.825, 0.85, 0.9, 0.95]:
        trial = run_trial(model=unregularised_model, prune_prop=prune_prop, lookahead=False, seed=seed)

        trial_results[prune_prop].append(trial)

    with open(f'si_finding_{seed}.json', 'w') as f:
        json.dump(trial_results, f, indent=4)

#### Test SITrainer Rashomon Iterations

In [48]:
trial_results = defaultdict(list)

for seed in range(3):
    model = pretrain(seed=seed)
    for rashomon_iters in [200, 500, 1000, 1500]:
        trial = run_trial(model=model, prune_prop=0.8, lookahead=False, rashomon_iters=rashomon_iters, seed=seed)
        
        trial_results[prune_prop].append(trial)

In [ ]:
x = sorted(trial_results.keys())

trial_y = [np.mean(trial_results[key]) for key in x]
trial_yerr = [np.std(trial_results[key]) for key in x]

# Create the plot
plt.figure(figsize=(10, 6))
plt.errorbar(x, trial_y, yerr=trial_yerr, fmt='o', capsize=5, color='green')
plt.xlabel("Frozen parameter proportion")
plt.ylabel("Mean Task 2 Accuracy")
plt.title("Top parameter freezing")
plt.grid(True)

### Hyperparam Tuning

In [ ]:
# HYPERPARAMS
# checkpoint
# n_iters
# primal_learning_rate
# dual_learning_rate
# batch_size
# learning rate
# pruning proportion
# penalty coefficient


def tuning_func(
    checkpoint: int = 100,
    n_iters: int = 300,
    min_acc_limit: float = 0.8,
    min_acc_increment: float = 0.05,
    primal_learning_rate: float = 0.5,
    dual_learning_rate: float = 0.1,
    learning_rate: float = 0.001,
    weight_decay: float = 0.01,
    l2_lambda: float = 0.01,
    unbias_lambda: float = 0.01,
    seed: int = 0,
    initial_epochs: int = 3,
    initial_batch_size: int = 128,
    device="cuda",
):
    SEED = seed
    train_tasks, val_tasks, test_tasks = get_mnist_tasks(seed=SEED)

    model = models.get_mnist_model(seed=SEED, device=device)

    l2 = L2Regulariser(lmbd=l2_lambda)
    unbias = UnbiasRegulariser(lmbd=unbias_lambda)
    regulariser = MultiRegulariser([unbias, l2])

    train_tasks, _ = zip(*[create_holdout_set(dataset) for dataset in train_tasks])
    trainer = SimpleTrainer(model, seed=SEED)
    trainer.train(
        train_tasks[0],
        val_tasks[0],
        epochs=initial_epochs,
        batch_size=initial_batch_size,
        regulariser=regulariser,
    )
    trainer.test(test_tasks[0:1])

    save_model = copy.deepcopy(trainer.model)

    si_trainer = SITrainer(
        save_model,
        checkpoint=checkpoint,
        n_iters=n_iters,
        min_acc_limit=min_acc_limit,
        min_acc_increment=min_acc_increment,
        primal_learning_rate=primal_learning_rate,
        dual_learning_rate=dual_learning_rate,
        paradigm="CIL",
        seed=SEED,
    )

    for i, (train, val, test) in enumerate(
        zip(train_tasks[1:], val_tasks[1:], test_tasks[1:]), start=1
    ):
        loader = DataLoader(
            train,
            batch_size=128,
            shuffle=True,
            generator=torch.Generator().manual_seed(SEED),
        )

        samples = next(iter(loader))
        si_trainer.compute_rashomon_set(
            test_tasks[i - 1], prune_prop=0.8, si_batch=samples
        )
        assert si_trainer.test(test_tasks[0 : i + 1])[-1][1] == 0, (
            "Prior last task performance needs to be zero."
        )
        si_trainer.train(
            train,
            val,
            epochs=20,
            batch_size=256,
            early_stopping=True,
            patience=10,
            lr=learning_rate,
            weigth_decay=weight_decay,
            regulariser=regulariser,
        )
        results = si_trainer.test(test_tasks[0 : i + 1])

        if not all([res[1] for res in results]):
            print("Catastrophic Forgetting occurred.")
            break

    return results

In [27]:
# It's always a good idea to have the latest version
!pip install wandb --upgrade
!pip install nbformat

import wandb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import numpy as np
import copy

# Log in to your W&B account
wandb.login()

In [ ]:
sweep_config = {
    "method": "bayes",  # Use Bayesian Optimization
    "metric": {"name": "final_total_accuracy", "goal": "maximize"},
    "parameters": {
        "checkpoint": {"values": [20, 50, 100, 150]},
        "n_iters": {"distribution": "q_uniform", "min": 200, "max": 500, "q": 50},
        "min_acc_limit": {"distribution": "uniform", "min": 0.7, "max": 0.95},
        "min_acc_increment": {"distribution": "uniform", "min": 0.01, "max": 0.1},
        "primal_learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.01,
            "max": 1.0,
        },
        "dual_learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.01,
            "max": 1.0,
        },
        "learning_rate": {
            "distribution": "log_uniform_values",
            "min": 0.0001,
            "max": 0.01,
        },
        "weight_decay": {
            "distribution": "log_uniform_values",
            "min": 1.0e-5,
            "max": 1.0e-2,
        },
        "l2_lambda": {
            "distribution": "log_uniform_values",
            "min": 1.0e-4,
            "max": 1.0e-1,
        },
        "unbias_lambda": {
            "distribution": "log_uniform_values",
            "min": 1.0e-4,
            "max": 1.0e-1,
        },
        "seed": {"values": list(range(100))},
        "initial_epochs": {"values": [3, 5, 10, 20, 50]},
        "initial_batch_size": {"values": [32, 64, 128, 256, 512, 1024]},
    },
}

In [ ]:
def train_one_run():
    # This function will be called by the W&B agent for each run.

    # Initialize a W&B run. W&B automatically fills wandb.config
    # with the hyperparameters for this run.
    with wandb.init() as run:
        config = wandb.config

        # Call your main logic with the hyperparameters
        final_results = tuning_func(
            checkpoint=config.checkpoint,
            n_iters=config.n_iters,
            min_acc_limit=config.min_acc_limit,
            min_acc_increment=config.min_acc_increment,
            primal_learning_rate=config.primal_learning_rate,
            dual_learning_rate=config.dual_learning_rate,
            learning_rate=config.learning_rate,
            weight_decay=config.weight_decay,
            l2_lambda=config.l2_lambda,
            unbias_lambda=config.unbias_lambda,
            device="cuda",
            seed=config.seed,
            initial_batch_size=config.initial_batch_size,
            initial_epochs=config.initial_epochs,
        )

        # Process and log the final metrics
        if final_results:
            accuracies = [res[1] for res in final_results]
            avg_accuracy = np.mean(accuracies)

            losses = [res[0] for res in final_results]
            avg_loss = np.mean(losses)

            wandb.log(
                {
                    "final_num_tasks": len(final_results),
                    "final_avg_accuracy": avg_accuracy,
                    "second_task_accuracy": accuracies[1] if len(accuracies) > 1 else 0,
                    "final_avg_loss": avg_loss,
                    "final_total_accuracy": np.sum(accuracies),
                }
            )
        else:
            # Log a failure if catastrophic forgetting happened early
            wandb.log({"final_avg_accuracy": 0.0, "final_total_accuracy": 0.0})

In [30]:
# 1. Initialize the sweep
# Replace "your-project-name" with a name for your W&B project
sweep_id = wandb.sweep(sweep_config, project="class_incremental-sweep")

# 2. Run the agent. This will execute your `train_one_run` function `count` times.
# This cell will run for a long time as it trains the model for each trial.
wandb.agent(sweep_id, function=train_one_run, count=100)